# motif_generator 使用说明

这个 notebook 说明如何使用 `src.motif_generator` 生成给定 `w,h` 下的 motif 组合。

设计原则：每个代码单元尽量自包含，可以单独运行。

## 1. 基本定义

一个 exact-box motif 的方框大小是 `w x h`：

- `w`：横向 plane/x 方向宽度
- `h`：纵向 y phase 高度
- 前 `w-1` 列是 planning columns
- 最后一列只是 target boundary

符号含义：

- `A = (dp=1, dy=0)`
- `B = (dp=1, dy=-1)`
- `C = (dp=1, dy=1)`
- `D = (dp=2, dy=0)`
- `-` 表示该节点不规划右邻居

当前模块统计的是 exact-box motif，不做全网平移等价合并。

In [ ]:
# 自包含导入单元：后面的任意代码单元如果单独运行，也可以先复制这段。
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.exact_box import (
    count_maximal_exact_box,
    enumerate_maximal_exact_box,
    enumerate_primitive_exact_box,
    motif_edges_as_user_ids,
    motif_from_columns,
    pretty_motif,
    smaller_repeat_factors,
)

print("motif_generator imported")

## 2. 计算某个 `w,h` 的数量

下面这个单元计算 `w=3,h=3` 的 maximal exact-box motif 和 primitive exact-box motif 数量。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.exact_box import (
    count_maximal_exact_box,
    enumerate_primitive_exact_box,
)

W = 3
H = 3
PHASE_COUNT = 36

maximal_count = count_maximal_exact_box(W, H, phase_count=PHASE_COUNT)
primitive_count = len(enumerate_primitive_exact_box(W, H, phase_count=PHASE_COUNT))

print(f"w={W}, h={H}")
print("maximal exact-box motifs:", maximal_count)
print("primitive exact-box motifs:", primitive_count)

## 3. 查看前几个 motif

`pretty_motif(motif)` 用列字符串描述局部连接，例如 `DAD | C--`。  
`motif_edges_as_user_ids(motif, row_pitch=36)` 会把局部连接转成 G60 风格的节点编号边列表。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.exact_box import (
    enumerate_maximal_exact_box,
    motif_edges_as_user_ids,
    pretty_motif,
)

W = 3
H = 3
ROW_PITCH = 36

motifs = enumerate_maximal_exact_box(W, H, phase_count=ROW_PITCH)
for idx, motif in enumerate(motifs[:10], start=1):
    print(f"#{idx:02d}", pretty_motif(motif))
    print("    edges:", ",".join(motif_edges_as_user_ids(motif, row_pitch=ROW_PITCH)))

## 4. Motif support representation

The recommended motif format is explicit `w/h/support`. Each support item is `(x, y, symbol)`: start from local coordinate `(x,y)` and connect by the offset represented by `symbol`.

This is close to `supp(M)` in the paper notation and is easier to read than `DAD | C--`.

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.support import (
    motif_support_from_dict,
    motif_support_label,
    motif_support_to_edge_records,
)
from src.motif_generator.module.tiling import node_id

motif = {
    "w": 3,
    "h": 3,
    "support": [
        (0, 0, "D"),
        (0, 1, "A"),
        (0, 2, "D"),
        (1, 0, "C"),
    ],
}

motif_support = motif_support_from_dict(motif)
edge_records = motif_support_to_edge_records(motif_support)

print("motif:", motif_support_label(motif_support))
for edge in edge_records:
    src = node_id(edge.src_col, edge.src_row, row_pitch=36)
    dst = node_id(edge.dst_col, edge.dst_row, row_pitch=36)
    print(f"{edge.symbol}: ({edge.src_col},{edge.src_row}) -> ({edge.dst_col},{edge.dst_row}) | user id {src}-{dst}")

## 5. 判断一个 motif 是否由更小 cell 重复而来

`smaller_repeat_factors(motif)` 会返回能够重复生成该 motif 的更小 exact-box 尺寸。  
如果返回空列表，那么这个 motif 在 exact-box 意义下是 primitive。

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.exact_box import (
    motif_from_columns,
    pretty_motif,
    smaller_repeat_factors,
)

candidates = [
    motif_from_columns(["AA", "AA"]),
    motif_from_columns(["DAD", "C--"]),
]

for motif in candidates:
    factors = smaller_repeat_factors(motif)
    print(pretty_motif(motif), "repeat_factors=", factors, "primitive=", len(factors) == 0)

## 6. 输出 CSV/JSON

如果只是想生成文件，可以直接运行 example 脚本。

In [ ]:
import subprocess
from pathlib import Path

PYTHON = Path(r"C:\ProgramData\miniconda3\envs\paper11\python.exe")
SCRIPT = Path(r"E:\paper11\generic\src\motif_generator\examples\enumerate_exact_box_motifs.py")
OUT_DIR = Path(r"E:\paper11\generic\src\motif_generator\examples\outputs\notebook_w3_h3")

cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--w", "3",
    "--h", "3",
    "--phase-count", "36",
    "--out-dir", str(OUT_DIR),
    "--limit-print", "5",
]

result = subprocess.run(cmd, text=True, capture_output=True, check=True)
print(result.stdout)
print("files:")
for path in sorted(OUT_DIR.glob("*")):
    print(" -", path)

## 7. Load a YAML motif, tile it on a full 2D grid, and draw it

This cell shows the recommended workflow: write `motif.w/h/support` in YAML, load it in Python, tile it on a full `p x n` grid, then write CSV/JSON/PNG outputs.

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.motif_generator.module.config_io import load_yaml_dict
from src.motif_generator.module.support import (
    motif_support_from_dict,
    motif_support_label,
    motif_support_to_edge_records,
)
from src.motif_generator.module.tiling import (
    draw_tiled_motif,
    tile_edge_records_on_grid,
    write_tiled_motif_outputs,
)

CONFIG = Path(r"E:\paper11\generic\src\motif_generator\examples\configs\dad_cxx_support.yaml")
OUT_DIR = Path(r"E:\paper11\generic\src\motif_generator\examples\outputs
otebook_yaml_tile_p18_n36_DAD_Cxx")

raw = load_yaml_dict(CONFIG)
grid = raw.get("grid", {})
tiling = raw.get("tiling", {})
motif_support = motif_support_from_dict(raw)

result = tile_edge_records_on_grid(
    p=int(grid.get("p", 18)),
    n=int(grid.get("n", 36)),
    motif_width=motif_support.w,
    motif_height=motif_support.h,
    local_edges=motif_support_to_edge_records(motif_support),
    horizontal_step=tiling.get("horizontal_step"),
    allow_vertical_overlap=bool(tiling.get("allow_vertical_overlap", True)),
    allow_clipped_right=bool(tiling.get("allow_clipped_right", True)),
)
write_tiled_motif_outputs(result, OUT_DIR)

png_path = OUT_DIR / "tiled_motif.png"
draw_tiled_motif(result, png_path)

print("motif:", motif_support.name or motif_support_label(motif_support))
print("accepted:", result.accepted_count)
print("rejected:", result.rejected_count)
print("edges:", result.edge_count)
print("png:", png_path)

try:
    from IPython.display import Image, display
    display(Image(filename=str(png_path)))
except Exception:
    pass

## 8. Run the YAML tiling example from command line

When you only need output files, run the example script directly. This command uses the same YAML file as the Python cell above.

In [ ]:
import subprocess
from pathlib import Path

PYTHON = Path(r"C:\ProgramData\miniconda3\envs\paper11\python.exe")
SCRIPT = Path(r"E:\paper11\generic\src\motif_generator\examples	ile_motif_on_grid.py")
CONFIG = Path(r"E:\paper11\generic\src\motif_generator\examples\configs\dad_cxx_support.yaml")
OUT_DIR = Path(r"E:\paper11\generic\src\motif_generator\examples\outputs
otebook_cli_yaml_tile_p18_n36_DAD_Cxx")

cmd = [
    str(PYTHON),
    str(SCRIPT),
    "--config", str(CONFIG),
    "--out-dir", str(OUT_DIR),
]

result = subprocess.run(cmd, text=True, capture_output=True, check=True)
print(result.stdout)
print("files:")
for path in sorted(OUT_DIR.glob("*")):
    print(" -", path)

## 9. Show the YAML motif with the shared 2D Topology Viewer

This cell uses `SatelliteTopology2DViewer`, the shared 2D topology display logic in this project.

In Jupyter, the Qt event loop must be enabled. The code includes `get_ipython().run_line_magic("gui", "qt5")`.

In [ ]:
import sys
from pathlib import Path

# Enable Qt event loop in Jupyter; equivalent to %gui qt5
try:
    get_ipython().run_line_magic("gui", "qt5")
except NameError:
    pass

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from PyQt5 import QtWidgets
from src.motif_generator.module.config_io import load_yaml_dict
from src.motif_generator.module.support import (
    motif_support_from_dict,
    motif_support_label,
    motif_support_to_edge_records,
)
from src.motif_generator.module.tiling import tile_edge_records_on_grid
from src.motif_generator.module.viewer_adapter import make_viewer_config, tiled_result_to_edge_table
from src.satellite_topology_viewer.module.base_viewer import SatelliteTopology2DViewer

CONFIG = Path(r"E:\paper11\generic\src\motif_generator\examples\configs\dad_cxx_support.yaml")
raw = load_yaml_dict(CONFIG)
grid = raw.get("grid", {})
tiling = raw.get("tiling", {})
viewer_cfg = raw.get("viewer", {})
motif_support = motif_support_from_dict(raw)

P = int(grid.get("p", 18))
N = int(grid.get("n", 36))
STEP = int(viewer_cfg.get("step", 1))

result = tile_edge_records_on_grid(
    p=P,
    n=N,
    motif_width=motif_support.w,
    motif_height=motif_support.h,
    local_edges=motif_support_to_edge_records(motif_support),
    horizontal_step=tiling.get("horizontal_step"),
    allow_vertical_overlap=bool(tiling.get("allow_vertical_overlap", True)),
    allow_clipped_right=bool(tiling.get("allow_clipped_right", True)),
)
edge_table = tiled_result_to_edge_table(result)
config = make_viewer_config(p=P, n=N, name="motif_topology")

app = QtWidgets.QApplication.instance()
if app is None:
    app = QtWidgets.QApplication([])

motif_label = motif_support.name or motif_support_label(motif_support)
viewer = SatelliteTopology2DViewer(
    config,
    steps=[STEP],
    edge_table=edge_table,
    window_title=f"motif topology {motif_label} at {STEP}s",
    group_data={},
    show_groups=False,
)
viewer.edge_width = float(viewer_cfg.get("edge_width", 0.045))
viewer.edge_alpha = int(viewer_cfg.get("edge_alpha", 190))
viewer.width_slider.setValue(int(max(8, min(50, round(viewer.edge_width * 1000)))))
viewer.alpha_slider.setValue(int(max(25, min(190, viewer.edge_alpha))))
viewer.resize(int(viewer_cfg.get("width", 1400)), int(viewer_cfg.get("height", 860)))
viewer.show()

_viewer_refs = globals().setdefault("_viewer_refs", [])
_viewer_refs.append(viewer)

print(f"viewer shown | motif={motif_label} | edges={edge_table.num_edges} | accepted={result.accepted_count}")
viewer